# Session 2A — DGAT model

Prepare graph inputs and inspect the DGAT architecture and five-term training objective without launching full training.

Work through the parts in order. Each part ends by writing its existing JSON checkpoint,
so the consolidation changes file organization without removing pause/resume milestones.


In [1]:
from pathlib import Path
import sys

# Locates the cloned tutorial in Google Colab after Session 0 setup.
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
colab_root = Path("/content/ECCB-2026-Tutorial/hands-on_tutorial")
if colab_root.is_dir():
    candidates.insert(0, colab_root.resolve())

for candidate in candidates:
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find hands-on_tutorial/. In Colab, run notebooks/session_00/00_colab_setup.ipynb first."
    )

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


Tutorial root: /Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial


## Session 2 · Part 1 — Prepare paired graphs for DGAT

**Goal:** turn aligned, normalized RNA and protein measurements into the two graph inputs used during
training. A spatial transcriptomics prediction sample has RNA only; paired spatial CITE-seq training
samples provide both modalities. This notebook uses the same official Tonsil RNA/ADT pair as Session 1,
so the tensor dimensions and graph construction correspond to the real tutorial dataset.


### 1. Process paired modalities


In [2]:
import numpy as np
import pandas as pd

from dgat_tutorial.data import find_dgat_h5ad_pair, load_tutorial_data
from dgat_tutorial.processing import build_dgat_graphs, knn_edge_index, process_modalities_official_dgat

pair = find_dgat_h5ad_pair(paths.raw_data)
dataset = load_tutorial_data(paths.raw_data)
processed = process_modalities_official_dgat(
    dataset.spots,
    dataset.transcripts,
    dataset.proteins,
    dgat_repo_dir=paths.root / "external" / "DGAT",
)
source = f"RNA={pair[0]}, ADT={pair[1]}"


/Users/zmengaf/miniconda3/envs/eccb-dgat-tutorial/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/zmengaf/miniconda3/envs/eccb-dgat-tutorial/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


/Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial/external/DGAT/utils/Preprocessing.py:51: ImplicitModificationWarning: Setting element `.layers['raw']` of view, initializing view as actual.
  adata.layers['raw'] = adata.X.copy()


/Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial/external/DGAT/utils/Preprocessing.py:52: ImplicitModificationWarning: Setting element `.layers['raw']` of view, initializing view as actual.
  pdata.layers['raw'] = pdata.X.copy()


### 2. Construct and save the RNA and protein graph inputs


In [3]:
# The official pipeline stores these as PyTorch Geometric HeteroData node/edge types.
# Here we expose the arrays first so dimensions and alignment are easy to inspect.
x_rna = processed.normalized_transcripts.to_numpy(dtype=np.float32)
x_protein = processed.normalized_proteins.to_numpy(dtype=np.float32)
graphs = build_dgat_graphs(
    processed.spots,
    processed.normalized_transcripts,
    processed.normalized_proteins,
)
rna_edge_index = graphs["rna_edge_index"]
protein_edge_index = graphs["protein_edge_index"]
spatial_edge_index = graphs["spatial_edge_index"]
print(
    "DGAT graphs = spatial 6-NN ∪ molecular 10-NN "
    f"(RNA PCA if >1500 genes). "
    f"spatial={spatial_edge_index.shape[1]}, "
    f"rna_union={rna_edge_index.shape[1]}, "
    f"protein_union={protein_edge_index.shape[1]}"
)

graph_summary = pd.DataFrame([
    {"graph": "RNA", "nodes": len(x_rna), "node_features": x_rna.shape[1], "directed_edges": rna_edge_index.shape[1]},
    {"graph": "protein", "nodes": len(x_protein), "node_features": x_protein.shape[1], "directed_edges": protein_edge_index.shape[1]},
])

display(graph_summary)

# Save the node-order mapping, graph summary, RNA union edges, and checkpoint metadata.
ids_path = paths.processed_data / "aligned_spot_ids.csv"
summary_path = paths.results / "session02_graph_input_summary.csv"
edge_path = paths.processed_data / "rna_spatial_edge_index.csv"
pd.DataFrame({"spot_id": processed.spots.index}).to_csv(ids_path, index=False)
graph_summary.assign(source=source).to_csv(summary_path, index=False)
pd.DataFrame(rna_edge_index.T, columns=["source_index", "target_index"]).to_csv(edge_path, index=False)
manifest = write_checkpoint(
    "2.1", [ids_path, summary_path, edge_path],
    summary={"source": source, "spots": len(processed.spots), "genes": x_rna.shape[1], "proteins": x_protein.shape[1]},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


DGAT graphs = spatial 6-NN ∪ molecular 10-NN (RNA PCA if >1500 genes). spatial=25146, rna_union=62477, protein_union=56469


,graph,nodes,node_features,directed_edges
0,RNA,4191,17434,62477
1,protein,4191,31,56469


Checkpoint written: /Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial/checkpoints/session_02/part_2_1.json


### Check

Both graphs must have the same node count and ordering in paired training data. Feature counts differ:
RNA nodes carry genes and protein nodes carry ADTs. Edge arrays use integer node positions and have
shape `2 × number_of_edges`.


## Session 2 · Part 2 — Create every DGAT model component

**Goal:** understand and instantiate the four modules that are trained jointly. This lesson follows the
official `Model/dgat.py`: separate RNA and protein graph-attention encoders, an RNA decoder, and a
branched protein decoder. The lightweight path prints the architecture and creates a figure; an optional
cell instantiates the real PyTorch modules when the official DGAT environment and repository are present.


### 1. Set dimensions from the processed data


In [6]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.teaching import (
    DGAT_PRETRAINED_GENE_COUNT,
    DGAT_PRETRAINED_PROTEIN_COUNT,
    official_dgat_component_table,
)

dataset = load_tutorial_data(paths.raw_data)
# Teaching dimensions follow the public pretrained ST checkpoint (Demo3 uses 11535 genes /
# 31 proteins). The Tonsil AnnData has a broader gene panel; inference zero-fills to the
# common-gene list via official fill_genes before protein_predict.
gene_list_path = paths.root / "external" / "DGAT" / "resources" / f"common_gene_{DGAT_PRETRAINED_GENE_COUNT}.txt"
protein_list_path = paths.root / "external" / "DGAT" / "resources" / f"common_protein_{DGAT_PRETRAINED_PROTEIN_COUNT}.txt"
if gene_list_path.exists() and protein_list_path.exists():
    common_genes = [line.strip() for line in gene_list_path.read_text().splitlines() if line.strip()]
    common_proteins = [line.strip() for line in protein_list_path.read_text().splitlines() if line.strip()]
    print(f"Using official common lists: {len(common_genes)} genes, {len(common_proteins)} proteins")
else:
    common_genes = list(dataset.transcripts.columns)[:DGAT_PRETRAINED_GENE_COUNT]
    common_proteins = list(dataset.proteins.columns)[:DGAT_PRETRAINED_PROTEIN_COUNT]
    print(
        "Official common_gene/protein lists not found under external/DGAT/resources/; "
        f"using the first {len(common_genes)} genes / {len(common_proteins)} proteins from Tonsil for the architecture table."
    )
HIDDEN_DIM = 1024  # official Train_and_Predict.py
component_table = official_dgat_component_table(len(common_genes), common_proteins, HIDDEN_DIM)
component_table


Using official common lists: 11535 genes, 31 proteins


,module,input,main_layers,output
0,RNA GAT encoder,"11535 RNA features + union(spatial 6-NN, RNA m...","GAT 2048 → GAT 1024 → GAT latent; residuals, L...",1024-D shared latent z_RNA
1,Protein GAT encoder,"31 proteins + union(spatial 6-NN, protein mole...","same GAT encoder design, separately learned we...",1024-D shared latent z_protein
2,RNA decoder,either shared latent,MLP 1024 → 512 → 1024 → 11535; residuals and L...,reconstructed/predicted RNA
3,Protein decoder,either shared latent,"shared MLP 1024 → 512 → 256, then one 256 → 64...",reconstructed/predicted protein panel


### Why does this RNA graph have 17,434 features while the pretrained model expects 11,535?

The two counts come from two different upstream DGAT training workflows:

- **17,434 genes:** `Demo1_Train.ipynb` trains on **Tonsil alone**. After
  `preprocess_train_list` applies CytAssist QC, 17,434 Tonsil genes remain. This tutorial uses that
  same single-sample preprocessing to demonstrate how a paired RNA/protein training graph is built.
- **11,535 genes:** `Pretrain_DGAT.ipynb` creates the public pretrained model from **six paired
  datasets**: Tonsil, Tonsil AddOns, Breast, Glioblastoma, PBC-PR_6835-5A, and PBC_PR_6837.
  `preprocess_train_list` first finds genes shared by the input datasets, applies the 2.5% gene
  prevalence and spot-level QC within each dataset (while retaining protein-encoding genes), and then
  intersects the post-QC gene sets again. That final sorted intersection contains 11,535 genes and is
  saved upstream as `common_gene_11535.txt`. The 31-protein panel is derived analogously as the common
  post-QC protein set.

Consequently, 11,535 is not an arbitrary truncation of the Tonsil matrix: it is the cross-dataset
feature vocabulary on which the public checkpoint was trained, and it fixes the RNA encoder's input
width and gene order. A 17,434-feature matrix cannot be passed directly to that encoder. During real
pretrained inference, `fill_genes` selects and reorders the RNA matrix to `common_gene_11535.txt` and
inserts zeros for missing panel genes; `preprocess_ST` then normalizes it before graph construction.
Thus, the graph above illustrates the one-sample paired-training workflow, not the exact matrix supplied
to the six-dataset public checkpoint.


### 2. Read the architecture from left to right

1. Each encoder performs three graph-attention stages with skip projections and LayerNorm.
2. After the first GAT stage, 16 feature-attention heads learn channel gates and their mean reweights the
   2048-dimensional representation.
3. Both encoders end in the same latent dimension so paired RNA and protein spots can be aligned.
4. The protein decoder shares two layers, then uses one output branch per protein. This lets proteins
   share signal while retaining protein-specific prediction heads.


#### Figure 6 — DGAT architecture and inference path


### 3. Instantiate the official modules (optional official environment)


In [8]:
# This is the exact constructor pattern used by the upstream training workflow.
# It executes only when external/DGAT and torch/torch_geometric are available.
dgat_repo = paths.root / "external" / "DGAT"
try:
    import torch
    sys.path.insert(0, str(dgat_repo))
    from Model.dgat import GATEncoder, Decoder_Protein, Decoder_mRNA
    OFFICIAL_READY = dgat_repo.is_dir()
except (ImportError, OSError):
    OFFICIAL_READY = False

if OFFICIAL_READY:
    encoder_rna = GATEncoder(in_channels=len(common_genes), hidden_dim=HIDDEN_DIM, dropout=0.3)
    decoder_rna = Decoder_mRNA(HIDDEN_DIM, len(common_genes), dropout=0)
    encoder_protein = GATEncoder(in_channels=len(common_proteins), hidden_dim=HIDDEN_DIM, dropout=0.3)
    decoder_protein = Decoder_Protein(HIDDEN_DIM, common_proteins, dropout=0)
    modules = {"RNA encoder": encoder_rna, "RNA decoder": decoder_rna,
               "protein encoder": encoder_protein, "protein decoder": decoder_protein}
    for name, module in modules.items():
        print(f"{name:18s}: {sum(p.numel() for p in module.parameters()):,} parameters")
else:
    print("Architecture lesson complete. To instantiate modules, use environment-dgat-cpu.yml and clone DGAT to external/DGAT.")


RNA encoder       : 87,158,784 parameters
RNA decoder       : 25,773,116 parameters
protein encoder   : 40,038,400 parameters
protein decoder   : 1,173,535 parameters


In [9]:
component_path = paths.results / "session02_dgat_components.csv"
component_table.to_csv(component_path, index=False)
manifest = write_checkpoint(
    "2.2", [component_path, architecture_path],
    summary={"modules": len(component_table), "official_modules_instantiated": OFFICIAL_READY}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


Checkpoint written: /Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial/checkpoints/session_02/part_2_2.json


### Check

Point to the exact inference route in the figure: **RNA graph → RNA encoder → z_RNA → protein
decoder**. The protein encoder and RNA decoder are training-time partners that make the shared latent
space learnable; they are not needed to impute protein on an RNA-only sample.


## Session 2 · Part 3 — Training objective (discussion; training skipped)

**Goal:** connect the four modules to five losses, four optimizers, backpropagation, evaluation, and
checkpoints — conceptually. **This tutorial does not train DGAT** (Colab / workshop compute limits).
We inspect the official objective and a pedagogical training step, then continue to pretrained
predictions in Part 4. Full training belongs in the upstream
[DGAT repository](https://github.com/osmanbeyoglulab/DGAT), not the live session.


### 1. Decompose the training objective


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.teaching import (
    DGAT_TRAIN_LOSS_WEIGHTS,
    official_dgat_loss_table,
    official_dgat_optimizer_table,
    weighted_training_objective,
)

loss_table = official_dgat_loss_table()
display(loss_table)
display(official_dgat_optimizer_table())
print(f"DGAT training coefficients (α,β,γ,δ,η): {DGAT_TRAIN_LOSS_WEIGHTS}")


The DGAT **training loop** uses five coefficients:

$$
(\alpha,\beta,\gamma,\delta,\eta)=(5,1,1,3,1).
$$

Its scalar training objective is

$$
\begin{aligned}
\mathcal{L}_{\mathrm{train}} ={}&
5\,\mathcal{L}_{\mathrm{RNA\ recon}}
+ \widetilde{\beta}\,\mathcal{L}_{\mathrm{protein\ recon}}
+ \widetilde{\gamma}\,\mathcal{L}_{\mathrm{alignment}} \\
&+ \widetilde{\delta}\,\mathcal{L}_{\mathrm{RNA}\rightarrow\mathrm{protein}}
+ 1\,\mathcal{L}_{\mathrm{protein}\rightarrow\mathrm{RNA}},
\end{aligned}
$$

Each corresponding effective coefficient—$\widetilde{\beta}$, $\widetilde{\gamma}$, or
$\widetilde{\delta}$—is replaced by zero when its unweighted loss is below $0.015$.
The RNA-reconstruction coefficient $\alpha=5$ and protein→RNA coefficient $\eta=1$
are not subject to this soft-zero rule.

The RNA→protein term directly trains the path used at inference. Always log the components
separately—a falling total can hide a failing task.


### 2. Inspect one real training step (pedagogical mirror)

This cell is for reading and discussion. It is **not** a complete training loop: it does not load
batches, run epochs, validate on a held-out sample, or save checkpoints. The figure below contrasts
the **recommended** held-out validate/save-best workflow (dashed) with what upstream
`Train_and_Predict.train(...)` currently exposes (solid boxes only through backprop + EB early stop).


In [13]:
try:
    import torch
except ImportError:
    torch = None

from dgat_tutorial.teaching import DGAT_LOSS_SOFT_THRESHOLD, DGAT_TRAIN_LOSS_WEIGHTS, apply_soft_loss_weights

def dgat_training_step(batch, modules, optimizers, rmse_loss, mse_loss, weights=DGAT_TRAIN_LOSS_WEIGHTS):
    # Readable mirror of the upstream DGAT optimization step (not invoked in the tutorial path).
    if torch is None:
        raise ImportError("torch is required to execute dgat_training_step; use eccb-dgat-official.")
    encoder_rna, decoder_rna, encoder_protein, decoder_protein = modules
    for optimizer in optimizers:
        optimizer.zero_grad()

    x_rna = batch["mRNA"].x
    e_rna = batch[("mRNA", "mRNA_knn", "mRNA")].edge_index
    x_protein = batch["protein"].x
    e_protein = batch[("protein", "protein_knn", "protein")].edge_index

    z_rna = encoder_rna(x_rna, e_rna)
    z_protein = encoder_protein(x_protein, e_protein)

    losses = {
        "rna_reconstruction": rmse_loss(decoder_rna(z_rna), x_rna),
        "protein_reconstruction": rmse_loss(decoder_protein(z_protein), x_protein),
        "latent_alignment": mse_loss(z_rna, z_protein),
        "protein_prediction": rmse_loss(decoder_protein(z_rna), x_protein),
        "rna_prediction": rmse_loss(decoder_rna(z_protein), x_rna),
    }
    effective = apply_soft_loss_weights(
        float(losses["protein_reconstruction"].detach()),
        float(losses["latent_alignment"].detach()),
        float(losses["protein_prediction"].detach()),
        weights=weights,
        soft_threshold=DGAT_LOSS_SOFT_THRESHOLD,
    )
    total = sum(weight * loss for weight, loss in zip(effective, losses.values()))
    total.backward()
    for module in modules:
        torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1.0)
    for optimizer in optimizers:
        optimizer.step()
    return {name: float(value.detach()) for name, value in losses.items()} | {"total": float(total.detach())}

print(f"torch={'unavailable' if torch is None else torch.__version__}")
print(f"Default train weights α,β,γ,δ,η = {DGAT_TRAIN_LOSS_WEIGHTS}; soft-threshold = {DGAT_LOSS_SOFT_THRESHOLD}")
print("dgat_training_step is defined for inspection only; the tutorial does not call it.")


torch=2.10.0
Default train weights α,β,γ,δ,η = (5.0, 1.0, 1.0, 3.0, 1.0); soft-threshold = 0.015
dgat_training_step is defined for inspection only; the tutorial does not call it.


#### Figure 7 — Upstream training path vs recommended validation loop


### 3. Training is skipped in this tutorial

Colab and the live workshop do **not** launch `Train_and_Predict.train(...)`. Reasons:

- official training needs multi-sample CITE-seq assets, substantial CPU/GPU time, and the separate
  DGAT dependency stack;
- the participant path evaluates **verified pretrained** Tonsil predictions instead.

Organizers who need to reproduce weights should use the upstream DGAT demos outside this tutorial.


In [15]:
print("Full DGAT training is skipped in the Colab / workshop path.")
print("Continue to Part 4 to load verified pretrained Tonsil predictions.")


Full DGAT training is skipped in the Colab / workshop path.
Continue to Part 4 to load verified pretrained Tonsil predictions.


### 4. What a trustworthy training run must save (for later study)

If you train DGAT later, save train/validation sample IDs, common gene/protein lists, preprocessing
parameters, random seed, per-epoch component losses, validation correlations, and all four state
dictionaries. Split by biological sample—not random spots—to avoid spatial and donor leakage. This
tutorial's committed predictions remain separate from observed evaluation proteins.


In [16]:
loss_path = paths.results / "session02_dgat_loss_terms.csv"
loss_table.to_csv(loss_path, index=False)
manifest = write_checkpoint(
    "2.3", [loss_path, workflow_path],
    summary={"loss_terms": len(loss_table), "full_training_executed": False, "runtime": "colab_skip_training"},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


Checkpoint written: /Users/zmengaf/Documents/ECCB tuto/hands-on_tutorial/checkpoints/session_02/part_2_3.json


### Check

Before moving on, explain why validation must hold out whole samples, which loss directly supervises
protein imputation, and which two modules are needed for RNA-only inference.
